# 02 — Fit Practice-Effect Slopes per Parcel

For each subject × task × contrast × parcel fit:
  `activation ~ centered_encounter`  (encounter centered at 3.0)

Then average slopes across subjects with a one-sample t-test against H₀: slope = 0.

**Outputs:**
- `processed_data/surface_indiv_slopes.pkl` — per-subject per-parcel regression results
- `processed_data/surface_avg_slopes.pkl` — group-averaged results

In [ ]:
import sys
import pickle
import numpy as np
import scipy.stats as stats
from pathlib import Path
from collections import defaultdict

try:
    _nb_dir = Path(__file__).resolve().parent
except NameError:
    _nb_dir = Path.cwd()
sys.path.insert(0, str(_nb_dir))
from config import TASKS, CONTRASTS, SUBJECTS, RT_CONTRAST, ENCOUNTER_CENTER, DATA_DIR

with open(DATA_DIR / 'surface_parcel_indiv.pkl', 'rb') as f:
    parcel_dict = pickle.load(f)

# Parcel names from any non-empty entry
for subj in SUBJECTS:
    for task in TASKS:
        for contrast in CONTRASTS[task]:
            if contrast == RT_CONTRAST:
                continue
            enc_data = parcel_dict.get(subj, {}).get(task, {}).get(contrast, {})
            if enc_data:
                parcel_names = list(enc_data.values())[0]['region'].tolist()
                break
        else:
            continue
        break

print(f'Loaded parcel_dict. Parcels: {len(parcel_names)}')

In [ ]:
def fit_parcel_slopes(parcel_dict, subject, task, contrast):
    """
    Fit linear regression (activation ~ centered_encounter) for each parcel.
    Returns dict keyed by parcel name. Missing encounters break the trajectory
    (no imputation at individual level, matching volumetric design).
    """
    enc_data = parcel_dict.get(subject, {}).get(task, {}).get(contrast, {})
    if not enc_data:
        return {}

    enc_keys_sorted = sorted(enc_data.keys())
    enc_numbers     = np.array([int(ek) for ek in enc_keys_sorted])
    enc_centered    = enc_numbers - ENCOUNTER_CENTER

    # (n_encounters, n_parcels)
    act_matrix = np.stack([enc_data[ek]['activation'].values for ek in enc_keys_sorted])

    results = {}
    for p_idx, parcel in enumerate(parcel_names):
        trajectory = act_matrix[:, p_idx]
        n = len(trajectory)

        if n < 2:
            continue

        slope, intercept, r_value, p_value, std_err = stats.linregress(enc_centered, trajectory)

        predicted  = slope * enc_centered + intercept
        residuals  = trajectory - predicted
        res_var    = np.sum(residuals**2) / max(n - 2, 1)
        res_std    = np.sqrt(res_var)
        rmse       = np.sqrt(np.mean(residuals**2))
        t_stat     = slope / std_err if std_err > 0 else 0.0

        abs_change     = trajectory[-1] - trajectory[0]
        traj_std       = np.std(trajectory)
        cohens_d       = abs_change / traj_std if traj_std > 0 else 0.0
        pct_change     = (abs_change / abs(trajectory[0])) * 100 if abs(trajectory[0]) > 0.001 else np.nan

        results[parcel] = {
            'beta_slope':           slope,
            'beta_intercept':       intercept,
            'std_error':            std_err,
            'ci_lower':             slope - 1.96 * std_err,
            'ci_upper':             slope + 1.96 * std_err,
            't_stat_slope':         t_stat,
            'p_value':              p_value,
            'df':                   n - 2,
            'significant_change':   p_value < 0.05,
            'r_squared':            r_value ** 2,
            'residual_variance':    res_var,
            'residual_std':         res_std,
            'rmse':                 rmse,
            'trajectory':           trajectory,
            'encounters_included':  enc_numbers.tolist(),
            'centered_predictor':   enc_centered.tolist(),
            'n_encounters_actual':  n,
            'mean_activation':      float(np.mean(trajectory)),
            'initial_activation':   float(trajectory[0]),
            'final_activation':     float(trajectory[-1]),
            'max_activation':       float(np.max(trajectory)),
            'min_activation':       float(np.min(trajectory)),
            'activation_range':     float(np.max(trajectory) - np.min(trajectory)),
            'trajectory_std':       float(traj_std),
            'absolute_change':      float(abs_change),
            'percent_change':       float(pct_change) if not np.isnan(pct_change) else np.nan,
            'cohens_d':             float(cohens_d),
        }
    return results

In [ ]:
# Individual slopes
indiv_slopes = defaultdict(lambda: defaultdict(lambda: defaultdict(dict)))

for subject in SUBJECTS:
    for task in TASKS:
        for contrast in CONTRASTS[task]:
            if contrast == RT_CONTRAST:
                continue
            indiv_slopes[subject][task][contrast] = fit_parcel_slopes(
                parcel_dict, subject, task, contrast
            )
    print(f'  Done: {subject}')

print('Individual slopes computed.')

In [ ]:
# Group averages — one-sample t-test on slopes across subjects
avg_slopes = defaultdict(lambda: defaultdict(dict))

for task in TASKS:
    for contrast in CONTRASTS[task]:
        if contrast == RT_CONTRAST:
            continue
        for parcel in parcel_names:
            subj_results = [
                indiv_slopes[s][task][contrast].get(parcel)
                for s in SUBJECTS
                if indiv_slopes[s][task][contrast].get(parcel) is not None
            ]
            if len(subj_results) < 2:
                continue

            slopes       = np.array([r['beta_slope']       for r in subj_results])
            intercepts   = np.array([r['beta_intercept']   for r in subj_results])
            r_squareds   = np.array([r['r_squared']        for r in subj_results])
            p_values     = np.array([r['p_value']          for r in subj_results])
            cohens_ds    = np.array([r['cohens_d']         for r in subj_results])
            init_acts    = np.array([r['initial_activation'] for r in subj_results])
            final_acts   = np.array([r['final_activation']   for r in subj_results])
            pct_changes  = np.array([r['percent_change']     for r in subj_results], dtype=float)
            max_acts     = np.array([r['max_activation']     for r in subj_results])
            min_acts     = np.array([r['min_activation']     for r in subj_results])
            act_ranges   = np.array([r['activation_range']   for r in subj_results])

            n = len(slopes)
            slope_mean = np.mean(slopes)
            slope_std  = np.std(slopes, ddof=1)
            slope_sem  = slope_std / np.sqrt(n)
            t_crit     = stats.t.ppf(0.975, df=n - 1)
            g_t, g_p   = stats.ttest_1samp(slopes, 0)

            # Trajectories: pad shorter ones with NaN
            traj_list = [r['trajectory'] for r in subj_results]
            max_len   = max(len(t) for t in traj_list)
            padded    = np.full((n, max_len), np.nan)
            for i, t in enumerate(traj_list):
                padded[i, :len(t)] = t
            traj_n   = np.sum(~np.isnan(padded), axis=0)
            traj_mean = np.nanmean(padded, axis=0)
            traj_std  = np.nanstd(padded, axis=0, ddof=1)
            traj_sem  = np.where(traj_n > 0, traj_std / np.sqrt(traj_n), np.nan)

            avg_slopes[task][contrast][parcel] = {
                'n_subjects':                        n,
                'slope_mean':                        slope_mean,
                'slope_std':                         slope_std,
                'slope_sem':                         slope_sem,
                'slope_ci_lower':                    slope_mean - t_crit * slope_sem,
                'slope_ci_upper':                    slope_mean + t_crit * slope_sem,
                'group_t_stat':                      g_t,
                'group_p_value':                     g_p,
                'group_significant':                 g_p < 0.05,
                'group_cohens_d':                    slope_mean / slope_std if slope_std > 0 else 0.0,
                'individual_slopes':                 slopes.tolist(),
                'individual_intercepts':             intercepts.tolist(),
                'individual_p_significant_proportion': float(np.mean(p_values < 0.05)),
                'positive_slope_proportion':         float(np.mean(slopes > 0)),
                'r_squared_mean':                    float(np.mean(r_squareds)),
                'r_squared_std':                     float(np.std(r_squareds, ddof=1)),
                'intercept_mean':                    float(np.mean(intercepts)),
                'intercept_sem':                     float(np.std(intercepts, ddof=1) / np.sqrt(n)),
                'initial_activation_mean':           float(np.mean(init_acts)),
                'initial_activation_sem':            float(np.std(init_acts, ddof=1) / np.sqrt(n)),
                'final_activation_mean':             float(np.mean(final_acts)),
                'final_activation_sem':              float(np.std(final_acts, ddof=1) / np.sqrt(n)),
                'percent_change_mean':               float(np.nanmean(pct_changes)),
                'cohens_d_mean':                     float(np.mean(cohens_ds)),
                'max_activation_mean':               float(np.mean(max_acts)),
                'min_activation_mean':               float(np.mean(min_acts)),
                'activation_range_mean':             float(np.mean(act_ranges)),
                'trajectory_mean':                   traj_mean,
                'trajectory_std':                    traj_std,
                'trajectory_sem':                    traj_sem,
                'trajectory_n_subjects':             traj_n,
            }

print('Group averages computed.')

In [ ]:
with open(DATA_DIR / 'surface_indiv_slopes.pkl', 'wb') as f:
    pickle.dump(dict(indiv_slopes), f)
with open(DATA_DIR / 'surface_avg_slopes.pkl', 'wb') as f:
    pickle.dump(dict(avg_slopes), f)

# Quick sanity check
example_task     = TASKS[0]
example_contrast = [c for c in CONTRASTS[example_task] if c != RT_CONTRAST][0]
example_parcel   = parcel_names[0]
eg = avg_slopes[example_task][example_contrast][example_parcel]
print(f'Example: {example_task} | {example_contrast} | {example_parcel}')
print(f'  slope_mean={eg["slope_mean"]:.4f}  group_p={eg["group_p_value"]:.4f}  significant={eg["group_significant"]}')
print(f'Saved to {DATA_DIR}')